# Lab Exercise: Train-Test Split & Validation Hazards
This notebook explores memorisation vs. generalisation, class stratification, preprocessing leakage, and the four major splitting failures (duplicates, grouping, time-series, and multiple test looks).

### Learning Objectives
1. **Memorisation Gaps**: Measure training vs. testing accuracy differences for high-capacity models.
2. **Split Lottery**: Quantify how much reported accuracy varies based purely on the random seed.
3. **Stratification**: Prevent target-imbalance drifts during partitioning.
4. **Group Leakage**: Prevent inflated scores when rows belong to correlated groups (e.g., patient IDs).
5. **Time-Series Split**: Contrast shuffled splits (interpolating) with chronological splits (extrapolating) on trending series.

## Setup & Environment Initialization

In [ ]:
import warnings
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
print("✓ Environment initialized successfully.")

## 1. Memorisation vs. Generalisation Gaps
Let's see how different models score on training data compared to test data. High-capacity models can easily reach 100% training accuracy through simple lookup, which is why a training evaluation is useless.

In [ ]:
X, y = load_breast_cancer(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)

print(f"   {'model':<28}{'train':>8}{'test':>8}{'gap':>8}")
for name, m in [("KNeighbors(n=1)", KNeighborsClassifier(1)),
                ("DecisionTree (unrestricted)", DecisionTreeClassifier(random_state=0)),
                ("RandomForest", RandomForestClassifier(random_state=0)),
                ("LogisticRegression", LogisticRegression(max_iter=5000))]:
    m.fit(Xtr, ytr)
    a, b = m.score(Xtr, ytr), m.score(Xte, yte)
    print(f"   {name:<28}{a:>8.4f}{b:>8.4f}{a - b:>8.4f}")

## 2. The Split Lottery
Let's observe how much reported model scores depend on nothing but the random seed we pass to `train_test_split`. We'll loop over 200 different random seeds across different row sizes.

In [ ]:
print(f"   {'rows':>7}{'test_size':>11}{'min':>8}{'max':>8}{'spread':>9}{'std':>8}")
for n in [80, 200, 569]:
    for ts in [0.2, 0.4]:
        s = []
        for seed in range(200):
            a, b, c, d = train_test_split(X[:n], y[:n], test_size=ts,
                                          random_state=seed, stratify=y[:n])
            s.append(make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
                     .fit(a, c).score(b, d))
        s = np.array(s)
        print(f"   {n:>7}{ts:>11.0%}{s.min():>8.3f}{s.max():>8.3f}"
              f"{s.max() - s.min():>9.3f}{s.std():>8.3f}")

## 3. Stratification vs. Imbalance Drift
When dealing with highly imbalanced classes, a simple random split can drift the proportion of positive cases in the test set. Let's verify this over 500 splits.

In [ ]:
rng = np.random.RandomState(0)
yi = (rng.rand(300) < 0.05).astype(int)
Xi = rng.normal(size=(300, 4))

print(f"STRATIFY — 300 rows, {yi.sum()} positives ({yi.mean():.1%}), 60-row test set")
for label, kw in [("without stratify", {}), ("with stratify=y", {"stratify": yi})]:
    c = np.array([int(train_test_split(Xi, yi, test_size=0.2, random_state=k, **kw)[3].sum())
                  for k in range(500)])
    print(f"   {label:<20} positives in test: min {c.min()}, max {c.max()},"
          f" mean {c.mean():.2f};  zero-positive splits: {int((c == 0).sum())}/500")

## 4. Grouped Data Leakage
Let's see why random splits fail on grouped measurements (e.g. repeated scans for the same patient). We'll build mock patient signatures where labels are patient-specific, meaning a generalizing model cannot beat chance (50%).

In [ ]:
P, R = 25, 20
g = np.repeat(np.arange(P), R)
sig = rng.normal(0, 3, (P, 5))
Xg = sig[g] + rng.normal(0, 1.0, (P * R, 5))
yg = rng.randint(0, 2, P)[g]

print("GROUPED DATA — random split vs. GroupShuffleSplit:")
rand = [RandomForestClassifier(random_state=0)
        .fit(*train_test_split(Xg, yg, test_size=0.3, random_state=k)[::2])
        .score(*train_test_split(Xg, yg, test_size=0.3, random_state=k)[1::2])
        for k in range(20)]
grp = []
for k in range(20):
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=k)
                  .split(Xg, yg, g))
    grp.append(RandomForestClassifier(random_state=0).fit(Xg[tr], yg[tr]).score(Xg[te], yg[te]))

print(f"   {'random split (same patient both sides)':<46}{np.mean(rand):>9.4f}")
print(f"   {'GroupShuffleSplit (patients kept whole)':<46}{np.mean(grp):>9.4f}")
print(f"   {'true ceiling (chance)':<46}{0.5:>9.4f}")

## 5. Time Series Splitting
Time-series models must be split chronologically. Shuffling time-series creates interpolation leakage, which reports perfect results on trending series where chronological predictions actually fail completely (due to lack of extrapolation capacity).

In [ ]:
t = np.arange(600)
print("TIME SERIES SKEW — shuffled vs. chronological splits:")
print(f"   {'series':<16}{'shuffled R2':>13}{'chronological R2':>18}")
for label, level in [
        ("mild trend", 0.05*t + 10*np.sin(2*np.pi*t/50) + rng.normal(0, 1, 600).cumsum()*0.3),
        ("strong trend", 1.0*t + 10*np.sin(2*np.pi*t/50) + rng.normal(0, 3, 600))]:
    lags = np.column_stack([np.roll(level, k) for k in range(1, 6)])
    Xs, ys = lags[10:], level[10:]
    a, b, c, d = train_test_split(Xs, ys, test_size=0.3, random_state=0)
    sh = r2_score(d, RandomForestRegressor(random_state=0).fit(a, c).predict(b))
    cut = int(len(Xs) * 0.7)
    ch = r2_score(ys[cut:], RandomForestRegressor(random_state=0)
                  .fit(Xs[:cut], ys[:cut]).predict(Xs[cut:]))
    print(f"   {label:<16}{sh:>13.4f}{ch:>18.4f}")

## Hands-On Programming Exercises

### Exercise 1: Evaluate Seed Variance
Compute the standard deviation of test accuracy over 50 different `random_state` seeds for the breast cancer dataset under `KNeighborsClassifier(n_neighbors=5)`. Report the min, max, and spread.

In [ ]:
# TODO: Loop over 50 seeds, store test accuracies, print metrics
accuracies = []

### Exercise 2: Grouped Data Split Verification
Write a script that takes a DataFrame with a `user_id` column, partitions it using `GroupShuffleSplit` (test_size=0.3), and programmatically asserts that no `user_id` is present in both training and testing datasets simultaneously.

In [ ]:
df_users = pd.DataFrame({
    'feature': rng.normal(size=100),
    'user_id': rng.choice([f"User_{i}" for i in range(10)], 100)
})

# TODO: Partition, and verify split integrity using sets
pass

### Exercise 3: Pipeline-Safe Scaling
Build a scikit-learn `Pipeline` containing `StandardScaler` and `LogisticRegression`. Split the dataset into train and test correctly (leak-free), fit the pipeline, and output the test accuracy. Add a comment explaining what statistic the scaler learned during the split.

In [ ]:
# TODO: Construct pipeline, perform leak-free fit, output score
pass